# Demonstrate accessing UM DYAMOND3 simulations from zarr on JASMIN object store: v5 version

### 24/4/25

* **Third look at simulations for WCRP Hackathon UK Node. v5 is/will be the final version of the global data**
* **Zoom level 10 (n2560/regional) or 9 (n1280) may not be complete**
* **Stores are not complete: some stores only have first 12 h, glm.n2560_RAL3p3 has most data but processing issues mean time data not contiguous**
* **However these stores will be filled in as the simulation data becomes avialable. We will let other nodes know when they are complete**
* Shows the hierarchy of simulations that will be available at the UK node, from global to regional.
* You can see the URLs which are active in the `cat` catalog.
* Contact mark.muetzelfeldt@reading.ac.uk for more info.

## Simulations

* glm: global model. n1280 is approx. 10 km res (stored at zoom 9), n2560 is 5 km (zoom 10). Regional simulations are at 4.4 km (zoom 10).
* Regional: Africa, South East Asia, South America, Cyclic Tropical Channel
* Settings:
    * CoMA9: CoMorph global,
    * RAL3: Regional Atmosphere Land 3
    * GAL9: Global Atmosphere Land 9
    * RAL3p3: RAL3.3
    * CoMA9_TBv1: CoMA9 TrailBlazer v1

## Technical

* All data stored as healpix, including regional.
* Regional simulations only store active chunks.
* Regional data necessarily has `nan`s to represent data outside the domain. This can cause issues when calculating domain means at different zooms. The `weights` field should help mitigate this (instructions to follow).
* There are two stores for each zoom level, one for `PT1H` (2D) and `PT3H` (3D) variables. All simulations are in the `sims` variable.
* Calling `ds = ds.compute()` downloads the data from JASMIN. This can be slow and/or fail with a server error. Try again if this happens.
* Can be run on JASMIN or anywhere else: call `Catalog(on_jasmin=True)` for JASMIN
* Tested using this Python conda env: https://github.com/digital-earths-global-hackathon/tools/blob/main/python_envs/environment.yaml (with some extra packages).
    * You can install with:
    * `wget https://raw.githubusercontent.com/digital-earths-global-hackathon/tools/refs/heads/main/python_envs/environment.yaml`
    * <edit last line of environment.yaml to be the name of your new env, e.g. hackathon_env>
    * `conda env create -f environment.yaml`
* Not all variables in the standard protocol are present - I have included those that are.
* I believe there is a plotting issue at lon=0 - and that data is OK.

## Issues

* CTC simulations where I think there is a genuine issue at lon=0
* Data not complete at zooms 9-0 (n2560/regional) or 8-0 (n1280), although empty zarr stores are present
* No data for most times for regional
* No zarr store for glm.n1280_CoMA9


In [1]:
import math as maths

import cartopy.crs as ccrs
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

import easygems.healpix as egh

/home/users/mmuetz/miniforge3/envs/hackathon_env/lib/python3.12/site-packages/pyproj/network.py:59: UserWarning: pyproj unable to set PROJ database path.
  _set_context_ca_bundle_path(ca_bundle_path)


In [2]:
ds = xr.open_dataset('http://hackathon-o.s3.jc.rl.ac.uk/sim-data/dev/v5.2/glm.n2560_RAL3p3/um.PT1H.hp_z10.zarr', engine='zarr')

In [3]:
ds

<xarray.Dataset> Size: 13TB
Dimensions:  (cell: 12582912, time: 9745)
Coordinates:
  * cell     (cell) int64 101MB 0 1 2 3 ... 12582908 12582909 12582910 12582911
    crs      float64 8B ...
  * time     (time) datetime64[ns] 78kB 2020-01-20 ... 2021-03-01
Data variables: (12/28)
    clivi    (time, cell) float32 490GB ...
    clt      (time, cell) float32 490GB ...
    clwvi    (time, cell) float32 490GB ...
    hflsd    (time, cell) float32 490GB ...
    hfssd    (time, cell) float32 490GB ...
    huss     (time, cell) float32 490GB ...
    ...       ...
    rsutcs   (time, cell) float32 490GB ...
    sftlf    (cell) float64 101MB ...
    tas      (time, cell) float32 490GB ...
    ts       (time, cell) float32 490GB ...
    uas      (time, cell) float32 490GB ...
    vas      (time, cell) float32 490GB ...
Attributes:
    Met Office DYAMOND3 simulations:  A group of experiments have been conduc...
    bounds:                           {'lower_left_lat': -90, 'lower_left_lon...
    latitiude_convention:             [-90, 90]
    longitude_convention:             [0, 360]
    regional:                         False
    simulation:                       glm.n2560_RAL3p3
    simulation_description:           The MetUM uses a regular lat-lon grid, ...

In [4]:
ds.orog

<xarray.DataArray 'orog' (cell: 12582912)> Size: 101MB
[12582912 values with dtype=float64]
Coordinates:
  * cell     (cell) int64 101MB 0 1 2 3 ... 12582908 12582909 12582910 12582911
    crs      float64 8B ...
Attributes:
    STASH:          [1, 0, 33]
    coarsened:      False
    grid_mapping:   healpix_nested
    healpix_zoom:   10
    long_name:      surface_altitude
    regrid_method:  easygems_delaunay
    source:         Data from Met Office Unified Model
    standard_name:  surface_altitude
    um_version:     13.5
    units:          m